## Working with APIs - GET/POST Requests and Parsing Responses

Modern data pipelines don't just read files - they often ingest data from real-time services, partner systems, or public APIs.

For example:

- Get daily weather or currency exchange data
- Fetch product or customer info from CRM APIs
- Ingest logs or alerts from observability APIs

So, understanding how to use Python's requests module to interact with APIs is a key skill in building ingestion pipelines.

### What Is an API?

An API (Application Programming Interface) is how two systems talk to each other.
 It's like saying: "Give me your data in JSON, I'll process it."
 
Most data APIs use:

HTTP Methods:

- GET → Fetch data
- POST → Send data or upload something

JSON format for responses
Endpoints (URLs) to access specific data

### 2. Setup - Import the requests Library

In [0]:
import requests

### 3. Example 1: Simple GET Request (JSONPlaceholder API)
A dummy API that returns fake JSON data - perfect for practice.

In [0]:
import requests

url = "https://jsonplaceholder.typicode.com/users"
response = requests.get(url)

print(response.status_code)   # Check if successful (200)
print(response.json()[0])     # Print first record

### 4. Parse and Load API Data into pandas

In [0]:
import pandas as pd

users = response.json()
df = pd.DataFrame(users)
print(df)
print(df[["id", "name", "email", "address"]].head())

### Flatten Nested JSON

In [0]:
df["city"] = df["address"].apply(lambda x: x["city"])
df_with_city = df[["id", "name", "email", "city"]]
print(df_with_city.head())

### 5. Example 2: Calling Weather API (Open-Meteo)

In [0]:
url = "https://api.open-meteo.com/v1/forecast"
params = {"latitude": 19.07, "longitude": 72.87, "current_weather": True}

response = requests.get(url, params=params)
data = response.json()
print(data["current_weather"])

In [0]:
weather_df = pd.DataFrame([data["current_weather"]])
print(weather_df)

###  6. Example 3: POST Request (Sending Data)

POST is used to send data to an API - for example, creating a new record.

In [0]:
url = "https://jsonplaceholder.typicode.com/posts"
payload = {
    "title": "Data Engineering Rocks!",
    "body": "Learning API handling with Python",
    "userId": 1
}

response = requests.post(url, json=payload)
print(response.status_code)
print(response.json())

###  7. Error Handling (Always a Must in ETL)

In [0]:
try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()   # Raises exception for 4xx/5xx
    data = response.json()
except requests.exceptions.Timeout:
    print(" Timeout - API took too long to respond.")
except requests.exceptions.HTTPError as err:
    print(" HTTP Error:", err)
except Exception as e:
    print(" Unexpected Error:", e)

### 8. Real-World Mini Project - Ingest Data from API → pandas → CSV

In [0]:
import requests, pandas as pd

# Step 1: Extract
url = "https://jsonplaceholder.typicode.com/comments"
response = requests.get(url)
data = response.json()

# Step 2: Transform
comments_df = pd.DataFrame(data)
comments_df = comments_df[["postId", "id", "name", "email", "body"]]

# Step 3: Load
comments_df.to_csv("/Volumes/thedataengineering_00/data/sampledata/comments_data.csv", index=False)

print(" Fetched", len(comments_df), "records from API and saved to comments_data.csv")